In [1]:
import sys, platform
ON_LINUXONE = platform.machine() == "s390x"
DATA_PATH = ("/home/ibm-user/transactions_history.csv" if ON_LINUXONE
             else "transactions_history.csv")
SEED = 42
TRAIN_END_YEAR = 2015         # train: up to 2015
VALID_YEARS = (2016, 2017)    # validation: compare models here
TEST_YEARS = (2018, 2019)     # test: use once in step 22

import pandas as pd
import numpy as np
import sklearn
import xgboost
import tensorflow as tf
import keras

print("Machine:", platform.machine(), "| Python:", sys.version.split()[0])

Machine: s390x | Python: 3.12.15


In [3]:
# 1. Load data
df = pd.read_csv(DATA_PATH)

# 2. Clean Amount: "$134.09" -> 134.09
df['Amount_num'] = df['Amount'].astype(str).str.replace('$', '', regex=False).astype(float)

# 3. Clean 'Is Fraud?' -> 1 / 0
# Bypass the object dtype check to support Pandas 3.x StringDtype
df['is_fraud'] = (df['Is Fraud?'].astype(str).str.strip().str.lower() == 'yes').astype(int)

# 4. Create single timestamp 'ts'
time_split = df['Time'].astype(str).str.split(':', expand=True)
df['hour'] = time_split[0].astype(int)
df['minute'] = time_split[1].astype(int)

df['ts'] = pd.to_datetime(dict(
    year=df['Year'],
    month=df['Month'],
    day=df['Day'],
    hour=df['hour'],
    minute=df['minute']
))

# 5. Sort chronologically
df = df.sort_values(by=['User', 'Card', 'ts']).reset_index(drop=True)

# 6. Assign train / valid / test splits by year
def assign_split(year):
    if year <= TRAIN_END_YEAR:
        return 'train'
    elif year in VALID_YEARS:
        return 'valid'
    elif year in TEST_YEARS:
        return 'test'
    else:
        return 'drop'

df['split'] = df['Year'].apply(assign_split)

# 7. Drop 2020+
df = df[df['split'] != 'drop'].reset_index(drop=True)

# 8. Print fraud counts to verify
fraud_counts = df.groupby('split')['is_fraud'].sum()
print("--- Fraud Count per Split ---")
print(fraud_counts)

--- Fraud Count per Split ---
split
test      60
train    178
valid     33
Name: is_fraud, dtype: int64


In [4]:
from sklearn.metrics import precision_score, recall_score, precision_recall_curve, auc, confusion_matrix

def report(y_true, scores, threshold=0.5):
    # Convert probabilities to binary predictions based on the threshold
    preds = (scores >= threshold).astype(int)
    
    # Calculate Precision and Recall
    precision = precision_score(y_true, preds, zero_division=0)
    recall = recall_score(y_true, preds, zero_division=0)
    
    # Calculate PR-AUC (Precision-Recall Area Under Curve)
    precision_curve, recall_curve, _ = precision_recall_curve(y_true, scores)
    pr_auc = auc(recall_curve, precision_curve)
    
    # Calculate False Positive Rate (FPR) and Alerts per 1,000
    tn, fp, fn, tp = confusion_matrix(y_true, preds).ravel()
    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0
    alerts_per_1000 = ((tp + fp) / len(y_true)) * 1000
    
    print(f"Precision:           {precision:.4f}")
    print(f"Recall:              {recall:.4f}")
    print(f"PR-AUC:              {pr_auc:.4f}")
    print(f"False Positive Rate: {fpr:.6f}")
    print(f"Alerts per 1,000:    {alerts_per_1000:.2f}")
    
    return precision, recall, pr_auc, fpr, alerts_per_1000

# Test the function with random data to verify it works
np.random.seed(SEED)
dummy_y = np.random.randint(0, 2, 10000)
dummy_scores = np.random.rand(10000)
print("--- Test Report (Random Scores) ---")
_ = report(dummy_y, dummy_scores, threshold=0.5)

--- Test Report (Random Scores) ---
Precision:           0.5040
Recall:              0.4965
PR-AUC:              0.4986
False Positive Rate: 0.486136
Alerts per 1,000:    491.30
